In [0]:
# Get parameters
dbutils.widgets.text("catalog", "bakehouse_analytics_dev")
dbutils.widgets.text("schema", "rnd")

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")

print(f"Running for catalog: {catalog}, schema: {schema}")

In [0]:
# Create schema if not exists
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")
print(f"Schema {catalog}.{schema} ready!")

In [0]:
# Read franchise data from samples
df_franchise = spark.sql("""
    SELECT
        n.n_name as country,
        s.s_acctbal as account_balance,
        CASE
            WHEN s.s_acctbal < 1000 THEN 'Small'
            WHEN s.s_acctbal BETWEEN 1000 AND 5000 THEN 'Medium'
            ELSE 'Large'
        END as franchise_size,
        COUNT(*) as franchise_count
    FROM samples.tpch.supplier s
    JOIN samples.tpch.nation n
        ON s.s_nationkey = n.n_nationkey
    GROUP BY
        n.n_name,
        s.s_acctbal,
        CASE
            WHEN s.s_acctbal < 1000 THEN 'Small'
            WHEN s.s_acctbal BETWEEN 1000 AND 5000 THEN 'Medium'
            ELSE 'Large'
        END
    ORDER BY country, franchise_size
""")

print("Franchise Analytics Data:")
df_franchise.show(10, truncate=False)

In [0]:
# Write to target table
df_franchise.write \
    .mode("overwrite") \
    .saveAsTable(f"{catalog}.{schema}.franchisee_analytics")

print(f"Success! Table {catalog}.{schema}.franchisee_analytics created!")

In [0]:
# Verify table was created
print("Verifying table contents:")
spark.read.table(f"{catalog}.{schema}.franchisee_analytics") \
    .show(10, truncate=False)